## Import

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import matplotlib.pyplot as plt


device = 'cuda:6'

%load_ext autoreload
%autoreload 2

## Dataset

In [2]:
from datasets import MultivariateStudentT

d = 64
dim_x = dim_y = d//2
rho = 0.7
df = 1
mean = np.zeros(dim_x + dim_y)
V = np.eye(dim_x)*rho
dispersion = np.eye(dim_x + dim_y)
dispersion[0:dim_x, :][:, dim_x:] = V
dispersion[dim_x:, :][:, 0:dim_x] = V


dataset = MultivariateStudentT(
        dim_x=dim_x,
        dim_y=dim_y,
        mean=mean,
        dispersion=dispersion,
        df=df)

X, Y = dataset.sample(10000)
X, Y = torch.Tensor(X).float().to(device), torch.Tensor(Y).float().to(device)

print(f'MI is {dataset.mutual_information(): 0.2f}')
print('X', X.shape, 'Y', Y.shape)

MI is  12.46
X torch.Size([10000, 32]) Y torch.Size([10000, 32])


## Hyperaparams

In [3]:
class Hyperparams(object):
    def __init__(self): 
        self.lr = 5e-4
        self.bs = 500
        self.wd = 1e-5
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]


## MI estimate

In [24]:
## Mutual information neural diffusion estimate (MINDE)
from estimators import MINDE

hyperparams.t_patience = 500
hyperparams.dim = d//2
hyperparams.device = device
hyperparams.importance_sampling = True

estimator = MINDE(hyperparams)

estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.mutual_information())
print('est MI:', estimator.MI(X, Y))

use ema: True bs: 500
finished: t= 0 loss= 1.5140767097473145 loss val= 1.5064144134521484 best val loss= 1.5064144134521484 best t= 0
finished: t= 76 loss= 0.3013058304786682 loss val= 0.2074793130159378 best val loss= 0.20604157447814941 best t= 69
finished: t= 152 loss= 0.25858640670776367 loss val= 0.20246733725070953 best val loss= 0.1763686239719391 best t= 122
finished: t= 228 loss= 0.16528701782226562 loss val= 0.19271203875541687 best val loss= 0.1763686239719391 best t= 122
finished: t= 304 loss= 0.23824021220207214 loss val= 0.2239876687526703 best val loss= 0.16708999872207642 best t= 237
finished: t= 380 loss= 0.2143818736076355 loss val= 0.22114376723766327 best val loss= 0.16708999872207642 best t= 237
finished: t= 456 loss= 0.15891289710998535 loss val= 0.1853039711713791 best val loss= 0.16708999872207642 best t= 237
finished: t= 532 loss= 0.20286722481250763 loss val= 0.21539250016212463 best val loss= 0.16708999872207642 best t= 237
finished: t= 608 loss= 0.210096374

In [23]:
## Mutual information neural estimate (MINE)
from estimators import MINE

estimator = MINE(architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.mutual_information())
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 0.00046128779649734497 loss val= 0.33852529525756836 best val loss= 0.33852529525756836 best t= 0
finished: t= 201 loss= -1.5618611574172974 loss val= 5.309891700744629 best val loss= -1.1048126220703125 best t= 200
finished: t= 402 loss= 1.0173189640045166 loss val= 2.590791702270508 best val loss= -1.2469735145568848 best t= 219


true MI: 12.456365411743121
est MI: 2.1795668601989746


In [5]:
## Vector copula estimation
from estimators import VCE

estimator = VCE(hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.mutual_information())
print('est MI:', estimator.MI(X, Y))


finished: t= 0 loss= 1.6292033195495605 loss val= 1.4205994606018066 best val loss= 1.4205994606018066 best t= 0
finished: t= 126 loss= 0.40834280848503113 loss val= 0.5395162105560303 best val loss= 0.5372331142425537 best t= 98
finished: t= 252 loss= 0.2565084397792816 loss val= 0.48442286252975464 best val loss= 0.45622706413269043 best t= 177
finished: t= 378 loss= 0.17583025991916656 loss val= 0.5130866169929504 best val loss= 0.44508129358291626 best t= 288


finished: t= 0 loss= 1.1165980100631714 loss val= 1.3056082725524902 best val loss= 1.3056082725524902 best t= 0
finished: t= 126 loss= 0.25151297450065613 loss val= 0.439542680978775 best val loss= 0.4333850145339966 best t= 118
finished: t= 252 loss= 0.19261297583580017 loss val= 0.45062220096588135 best val loss= 0.40384912490844727 best t= 224
finished: t= 378 loss= 0.6014723181724548 loss val= 0.7315284013748169 best val loss= 0.40269389748573303 best t= 277


finished: t= 0 loss= 622.6549072265625 loss val= 635.3462524

In [13]:
## MI estimate via flows (MIENF)
from estimators import MIENF

estimator = MIENF(hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.mutual_information())
print('est MI:', estimator.MI(X, Y))


MIENF (K=1), joint learning True 

finished: t= 0 loss= 4034.281982421875 loss val= 3123.248046875 best val loss= 3123.248046875 best t= 0
finished: t= 101 loss= -119.20645904541016 loss val= 1374249.0 best val loss= 896.236572265625 best t= 1


true MI: 12.456365411743121
est MI: -299.5057067871094
